<a href="https://colab.research.google.com/github/aditi-govindu/gemini-recipe-agent/blob/main/recipe_agent_gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# About
This colab is an introduction to a RAG based ReAct agent that looks at a recipe book, reasons the best course of action and generates a response.

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to aistudio.google.com
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0



In [ ]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

API key loaded from Colab Secrets.


In [ ]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai --quiet
print("Done.")

Done.


In [ ]:
# @title Create a database of recipes.
RECIPES = {
    "bombe_alaska": (
            "1 pint vanilla ice cream softened"
            "1 pint raspberry sorbet softened"
            "1 pint chocolate ice cream softened"
            "1 prepared 9-inch sponge cake or genoise layer"
            "4 large egg whites"
            "1/2 cup granulated sugar"
            "Pinch of cream of tartar"
            "Line a 2-quart bowl with plastic wrap leaving an overhang. Spoon vanilla ice cream into the bowl spreading evenly. Freeze for 30 minutes."
            "Repeat with raspberry sorbet then chocolate ice cream freezing each layer for 30 minutes."
            "Place the cake layer on top of the chocolate ice cream pressing gently. Cover with plastic wrap and freeze overnight."
            "Preheat oven to 500°F (260°C). In a large bowl whisk egg whites until foamy. Gradually add sugar and cream of tartar whisking until stiff glossy peaks form (meringue)."
            "Unmold the ice cream dome onto an oven-safe serving plate. Remove plastic wrap. Spread meringue all over the ice cream dome ensuring it's completely covered and sealed to the plate."
            "Bake for 3-5 minutes or until meringue is golden brown. Serve immediately."
            ),
    "fried_rice": (
            "3 cups cooked day-old white rice"
            "2 tbsp vegetable oil"
            "1/2 cup diced carrots"
            "1/2 cup frozen peas"
            "1/2 cup chopped onion"
            "2 cloves garlic minced"
            "2 large eggs beaten"
            "2 tbsp soy sauce"
            "1 tbsp oyster sauce (optional)"
            "1 tsp sesame oil"
            "Salt and pepper to taste"
            "Green onions chopped for garnish"
            "Heat vegetable oil in a large skillet or wok over medium-high heat."
            "Add carrots peas and onion. Cook stirring frequently for 3-5 minutes until vegetables are tender-crisp."
            "Add minced garlic and cook for another minute until fragrant."
            "Push vegetables to one side of the skillet. Pour beaten eggs into the empty side and scramble until cooked through. Break into small pieces."
            "Add cooked rice to the skillet. Pour soy sauce oyster sauce (if using) and sesame oil over the rice. Stir everything together breaking up any clumps of rice."
            "Cook for 5-7 minutes stirring occasionally until the rice is heated through and slightly browned."
            "Season with salt and pepper to taste. Garnish with chopped green onions before serving."
    ),
    "butter_chicken": (
            "1.5 lbs boneless skinless chicken thighs cut into 1-inch pieces"
            "1/2 cup plain yogurt"
            "1 tbsp lemon juice"
            "2 tsp ground cumin"
            "2 tsp garam masala"
            "1 tsp turmeric powder"
            "1 tsp chili powder (or paprika for milder taste)"
            "1/2 tsp salt"
            "2 tbsp butter"
            "1 large onion finely chopped"
            "2 cloves garlic minced"
            "1 tbsp grated fresh ginger"
            "1 (14.5 ounce) can crushed tomatoes"
            "1/2 cup heavy cream"
            "1/4 cup chopped fresh cilantro for garnish"
            "In a large bowl combine chicken pieces with yogurt lemon juice cumin garam masala turmeric chili powder and salt. Mix well to coat the chicken. Marinate for at least 30 minutes or preferably 2-4 hours in the refrigerator."
            "Melt butter in a large skillet or Dutch oven over medium heat. Add chopped onion and cook until softened and translucent about 5-7 minutes."
            "Add minced garlic and grated ginger cook for 1 minute until fragrant."
            "Add the marinated chicken to the skillet. Cook stirring occasionally until browned on all sides about 8-10 minutes."
            "Stir in the crushed tomatoes. Bring to a simmer then reduce heat to low cover and cook for 15-20 minutes or until chicken is cooked through."
            "Stir in the heavy cream. Cook for another 5 minutes uncovered until the sauce thickens slightly. Adjust seasoning if needed."
            "Serve hot over rice or with naan bread garnished with fresh cilantro."
    ),
    "alfredo_pasta": (
            "1 lb fettuccine pasta"
            "1/2 cup unsalted butter"
            "2 cups heavy cream"
            "1.5 cups freshly grated Parmesan cheese"
            "1/4 tsp ground nutmeg (optional)"
            "Salt and freshly ground black pepper to taste"
            "Fresh parsley chopped for garnish"
            "Cook fettuccine according to package directions until al dente. Reserve 1 cup of pasta water before draining."
            "While pasta cooks melt butter in a large skillet or saucepan over medium-low heat."
            "Pour in heavy cream and bring to a gentle simmer stirring occasionally. Do not boil."
            "Reduce heat to low. Gradually whisk in the Parmesan cheese until melted and the sauce is smooth. If the sauce is too thick add a little reserved pasta water until desired consistency is reached."
            "Season with nutmeg (if using) salt and freshly ground black pepper. Taste and adjust seasoning."
            "Add the drained fettuccine to the sauce and toss to coat evenly."
            "Serve immediately garnished with chopped fresh parsley."
    )
}

In [ ]:
# @title Define a tool
def search_recipes(query: str) -> str:
    """Search the cookbook collection for recipes relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in RECIPES.items():
        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content.lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant recipes found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Quick test
print(search_recipes("fresh parsley"))


[Source: alfredo_pasta]
1 lb fettuccine pasta1/2 cup unsalted butter2 cups heavy cream1.5 cups freshly grated Parmesan cheese1/4 tsp ground nutmeg (optional)Salt and freshly ground black pepper to tasteFresh parsley chopped for garnishCook fettuccine according to package directions until al dente. Reserve 1 cup of pasta water before draining.While pasta cooks melt butter in a large skillet or saucepan over medium-low heat.Pour in heavy cream and bring to a gentle simmer stirring occasionally. Do not boil.Reduce heat to low. Gradually whisk in the Parmesan cheese until melted and the sauce is smooth. If the sauce is too thick add a little reserved pasta water until desired consistency is reached.Season with nutmeg (if using) salt and freshly ground black pepper. Taste and adjust seasoning.Add the drained fettuccine to the sauce and toss to coat evenly.Serve immediately garnished with chopped fresh parsley.

[Source: butter_chicken]
1.5 lbs boneless skinless chicken thighs cut into 1-inc

In [ ]:
# @title Ask gemini to search for something
import google.generativeai as genai
import time

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

model = genai.GenerativeModel(
    model_name="gemini-3.5-flash-lite",
    tools=[search_recipes],
    system_instruction=(
        '''
        You are a chef training at a Michelin star restaurant.
        When you need specific information about temperatures,
        ingredients to use, cook time etc. use the search_documents tool.
        Only answer based on what you find in the documents.
        If the documents do not contain the answer, say so clearly.
        '''
    )
)

print("Model ready:", model.model_name)

Model ready: models/gemini-3.5-flash-lite


In [ ]:
import time
from google import genai
from google.genai import types

client = genai.Client()  # reads GEMINI_API_KEY / GOOGLE_API_KEY from env

search_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="search_recipes",
        description="Search the cookbook to find most the relevant recipe to cook.",
        parameters={
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    )
])

config = types.GenerateContentConfig(
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)


def send_with_retry(contents, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=model.model_name,
                contents=contents, config=config
            )
        except Exception as e:
            if "429" in str(e) and attempt < max_retries - 1:
                wait = 20 * (attempt + 1)
                print(f"  Rate limit hit, waiting {wait}s...")
                time.sleep(wait)
            else:
                raise


def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\n" + "-" * 50)
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)
        model_content = response.candidates[0].content

        # Append the model's turn UNMODIFIED, so thought signatures are preserved
        contents.append(model_content)

        calls = [p.function_call for p in model_content.parts if p.function_call]
        if not calls:
            print(f"\nFinal answer:\n{response.text}")
            return response.text

        # Run every requested tool call, reply with all results in ONE user turn
        result_parts = []
        for call in calls:
            args = dict(call.args)
            print(f"Step {step}: model calls {call.name}({args})")
            result = search_recipes(**args)
            print(f"  Result preview: {str(result)[:100]}...")
            result_parts.append(
                types.Part.from_function_response(
                    name=call.name, response={"result": result}
                )
            )
        contents.append(types.Content(role="user", parts=result_parts))

    return "(Stopped: hit max_steps)"


_ = run_agent("What can I cook with vanilla and caramel")

Question: What can I cook with vanilla and caramel
--------------------------------------------------
Step 1: model calls search_recipes({'query': 'vanilla caramel'})
  Result preview: [Source: bombe_alaska]
1 pint vanilla ice cream softened1 pint raspberry sorbet softened1 pint choco...

Final answer:
Here are a few delicious ideas for desserts and treats you can make using both **vanilla** and **caramel**:

1. **Salted Caramel Vanilla Milkshake / Frappe**
   * **How to make it:** Blend vanilla ice cream, a splash of milk, a teaspoon of vanilla extract (or vanilla bean paste), and a generous drizzle of caramel sauce. Top with whipped cream and a sprinkle of sea salt.

2. **Caramel Vanilla Custard / Crème Caramel**
   * **How to make it:** Make a classic custard base flavored with vanilla bean or vanilla extract, poured over a layer of homemade melted caramel at the bottom of ramekins, then baked in a water bath until set.

3. **Vanilla Cake with Caramel Buttercream**
   * **How to mak

In [ ]:
questions = [
    "What is the best chicken dish on our menu?",
    "Does our restaurant's pasta use ravioli or linguine?",
    "How long must the rice be cooked for",
    # This question is not answered by our database.
    "How many customers eat our food everyday?",
    # This question is part of the general knowledge of the model.
    "What is the most famous food item in Japan?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # stay within the free-tier rate limit.
    print("\n" + "=" * 60)
    run_agent(q)



Question: How many customers eat our food everyday?
--------------------------------------------------
Step 1: model calls search_recipes({'query': 'customers eat everyday'})
  Result preview: [Source: fried_rice]
3 cups cooked day-old white rice2 tbsp vegetable oil1/2 cup diced carrots1/2 cu...

Final answer:
I don't have access to information about customer numbers or restaurant statistics. My database only contains recipes and cooking instructions!

Question: What is the most famous food item in Japan?
--------------------------------------------------
Step 1: model calls search_recipes({'query': 'sushi ramen tempura'})
  Result preview: No relevant recipes found for that query....

Final answer:
While Japan has many world-renowned culinary treasures—such as **Ramen**, **Tempura**, and **Wagyu beef**—**Sushi** is widely considered the most famous and iconic Japanese food item globally. 

Sushi is celebrated for its artistry, fresh ingredients, and cultural significance, representin

# Next steps:
* Save a copy of this code to your GitHub repo.
* Create a tool to recommend ingredient substitutions for recipes.
* Push the new tool to your colab and v2 to the same GitHub repo.